In [7]:
import zipfile
import networkx as nx
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# REPRODUCIBILITY
# -----------------------------
np.random.seed(7)
random.seed(7)

# -----------------------------
# 1. LOAD GRAPH
# -----------------------------
def load_graph(file_path):
    G = nx.Graph()

    with zipfile.ZipFile(file_path, 'r') as z:
        for filename in z.namelist():
            if filename.endswith(".txt"):
                with z.open(filename) as f:
                    for line in f:
                        try:
                            u, v = map(int, line.decode().strip().split())
                            G.add_edge(u, v)
                        except:
                            continue
                break

    print("Original Graph:")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

    return G


# -----------------------------
# 2. SAMPLE GRAPH (SPEED)
# -----------------------------
def sample_graph(G, sample_size=15000):
    nodes = list(G.nodes())
    sampled_nodes = set(random.sample(nodes, sample_size))
    G_sub = G.subgraph(sampled_nodes).copy()

    print("\nSampled Graph:")
    print("Nodes:", G_sub.number_of_nodes())
    print("Edges:", G_sub.number_of_edges())

    return G_sub


# -----------------------------
# 3. TRAFFIC SIMULATION
# -----------------------------
def simulate_traffic(G, timesteps=50, mu=10, sigma=2):
    traffic = {}
    for u, v in G.edges():
        edge = (min(u, v), max(u, v))
        traffic[edge] = np.random.normal(mu, sigma, timesteps)
    return traffic


# -----------------------------
# 4. ATTACK MODEL (REALISTIC)
# -----------------------------
def inject_attack(G, traffic, attack_strength=40, duration=8, k=5):
    targets = sorted(G.degree, key=lambda x: x[1], reverse=True)[:k]
    attacked_edges = []

    for node, _ in targets:
        edges = list(G.edges(node))
        if not edges:
            continue

        sampled_edges = random.sample(edges, max(1, len(edges)//2))

        for u, v in sampled_edges:
            edge = (min(u, v), max(u, v))
            loads = traffic[edge].copy()

            t_attack = random.randint(10, 30)

            for t in range(t_attack, min(t_attack + duration, len(loads))):
                loads[t] += attack_strength * np.random.uniform(0.7, 1.2)

            traffic[edge] = loads
            attacked_edges.append(edge)

    return attacked_edges


# -----------------------------
# 5. ENTROPY (FIXED)
# -----------------------------
def compute_entropy_weights(G, traffic):
    weights = {}

    for u, v in G.edges():
        edge = (min(u, v), max(u, v))
        loads = traffic[edge]

        var = np.var(loads)
        temporal = np.mean(np.abs(np.diff(loads)))

        baseline = 1.0  # 🔥 prevents perfect avoidance
        entropy = np.log(1 + var + 0.5 * temporal)

        weights[edge] = min(baseline + entropy, 5.0)

    return weights


# -----------------------------
# 6. ASSIGN WEIGHTS
# -----------------------------
def assign_weights(G, weights):
    for u, v in G.edges():
        edge = (min(u, v), max(u, v))
        G[u][v]['weight'] = weights[edge]


# -----------------------------
# 7. ROUTING EVALUATION
# -----------------------------
def evaluate_routing(G, weights, attacked_edges, num_pairs=120):
    nodes = list(G.nodes())

    results = {
        "shortest_path_length": [],
        "entropy_path_length": [],
        "shortest_entropy": [],
        "entropy_entropy": [],
        "shortest_attack_hits": [],
        "entropy_attack_hits": []
    }

    assign_weights(G, weights)
    attacked_set = set(attacked_edges)

    high_degree_nodes = [n for n, _ in sorted(G.degree, key=lambda x: x[1], reverse=True)[:50]]

    def path_entropy(path):
        total = 0
        for i in range(len(path)-1):
            edge = (min(path[i], path[i+1]), max(path[i], path[i+1]))
            total += weights[edge]
        return total

    def attack_count(path):
        count = 0
        for i in range(len(path)-1):
            edge = (min(path[i], path[i+1]), max(path[i], path[i+1]))
            if edge in attacked_set:
                count += 1
        return count

    for _ in range(num_pairs):
        if len(nodes) < 2:
            continue

        # 🔥 force harder routing scenarios
        if random.random() < 0.5:
            src = random.choice(high_degree_nodes)
            dst = random.choice(nodes)
        else:
            src = random.choice(nodes)
            dst = random.choice(high_degree_nodes)

        if src == dst:
            continue

        try:
            sp = nx.bidirectional_shortest_path(G, src, dst)
            ep = nx.bidirectional_dijkstra(G, src, dst, weight='weight')[1]

            results["shortest_path_length"].append(len(sp) - 1)
            results["entropy_path_length"].append(len(ep) - 1)

            results["shortest_entropy"].append(path_entropy(sp))
            results["entropy_entropy"].append(path_entropy(ep))

            results["shortest_attack_hits"].append(attack_count(sp))
            results["entropy_attack_hits"].append(attack_count(ep))

        except:
            continue

    return results


# -----------------------------
# 8. SUMMARY
# -----------------------------
def summarize(results):
    print("\n--- RESULTS ---")

    print(f"Avg Path Length (Shortest): {np.mean(results['shortest_path_length']):.2f}")
    print(f"Avg Path Length (Entropy): {np.mean(results['entropy_path_length']):.2f}")

    print(f"Avg Entropy (Shortest): {np.mean(results['shortest_entropy']):.2f}")
    print(f"Avg Entropy (Entropy): {np.mean(results['entropy_entropy']):.2f}")

    print(f"Attack Hits (Shortest): {np.mean(results['shortest_attack_hits']):.4f}")
    print(f"Attack Hits (Entropy): {np.mean(results['entropy_attack_hits']):.4f}")


# -----------------------------
# 9. PLOTS
# -----------------------------
def plot_results(results):
    plt.figure()
    plt.hist(results["shortest_path_length"], bins=20, alpha=0.7, label="Shortest Path")
    plt.hist(results["entropy_path_length"], bins=20, alpha=0.7, label="Entropy Routing")
    plt.legend()
    plt.title("Path Length Distribution")
    plt.savefig("path_length.png")
    plt.close()

    plt.figure()
    plt.hist(results["shortest_entropy"], bins=20, alpha=0.7, label="Shortest Path")
    plt.hist(results["entropy_entropy"], bins=20, alpha=0.7, label="Entropy Routing")
    plt.legend()
    plt.title("Path Entropy Comparison")
    plt.savefig("entropy_comparison.png")
    plt.close()

    plt.figure()
    values = [
        np.mean(results["shortest_attack_hits"]),
        np.mean(results["entropy_attack_hits"])
    ]
    labels = ["Shortest Path", "Entropy Routing"]

    plt.bar(labels, values)
    plt.title("Attack Exposure Comparison")
    plt.savefig("attack_exposure.png")
    plt.close()


# -----------------------------
# 10. EXPORT
# -----------------------------
def export_results(results):
    pd.DataFrame(results).to_csv("routing_results.csv", index=False)


# -----------------------------
# MAIN
# -----------------------------
if __name__ == "__main__":
    file_path = "twitter_combined.txt.zip"

    print("Loading graph...")
    G = load_graph(file_path)

    print("\nSampling graph...")
    G = sample_graph(G, sample_size=15000)

    print("\nSimulating traffic...")
    traffic = simulate_traffic(G)

    print("Injecting attack...")
    attacked_edges = inject_attack(G, traffic)

    print("Computing entropy...")
    weights = compute_entropy_weights(G, traffic)

    print("Evaluating routing...")
    results = evaluate_routing(G, weights, attacked_edges)

    summarize(results)
    plot_results(results)
    export_results(results)

    print("\nSaved:")
    print("- path_length.png")
    print("- entropy_comparison.png")
    print("- attack_exposure.png")
    print("- routing_results.csv")

Loading graph...
Original Graph:
Nodes: 81306
Edges: 1342310

Sampling graph...

Sampled Graph:
Nodes: 15000
Edges: 43597

Simulating traffic...
Injecting attack...
Computing entropy...
Evaluating routing...

--- RESULTS ---
Avg Path Length (Shortest): 4.12
Avg Path Length (Entropy): 4.14
Avg Entropy (Shortest): 11.97
Avg Entropy (Entropy): 11.43
Attack Hits (Shortest): 0.2262
Attack Hits (Entropy): 0.0476

Saved:
- path_length.png
- entropy_comparison.png
- attack_exposure.png
- routing_results.csv
